# 02 – Data Cleaning

**Goal:** Validate the raw data, handle missing values and duplicates, and save a cleaned CSV.

This notebook does not invent employee activity values. Numeric missing values are filled with the column median; categorical missing values would be filled with the mode.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "dataset"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np

df = pd.read_csv(DATA_DIR / "employee_activity.csv")
print("Original shape:", df.shape)
df.head()

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()
print("Shape after duplicate removal:", df.shape)

In [ ]:
# Check missing values
df.isnull().sum()

In [ ]:
# Fill missing numeric values with the median
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill missing text/category values with the most frequent value
categorical_cols = df.select_dtypes(exclude=np.number).columns
for col in categorical_cols:
    if df[col].isnull().any():
        mode_values = df[col].mode()
        if not mode_values.empty:
            df[col] = df[col].fillna(mode_values.iloc[0])

print("Missing values after cleaning:")
print(df.isnull().sum())

In [ ]:
# Validate expected binary columns
for col in ["login_after_hours", "usb_connected", "threat_label"]:
    print(col, sorted(df[col].dropna().unique()))

In [ ]:
# Validate non-negative activity counts
count_cols = [
    "files_downloaded",
    "failed_logins",
    "sensitive_file_access"
]
for col in count_cols:
    print(f"{col}: minimum = {df[col].min()}")

In [ ]:
# Save cleaned dataset
clean_path = DATA_DIR / "employee_activity_cleaned.csv"
df.to_csv(clean_path, index=False)
print("Saved:", clean_path)

## Important note

Do not automatically delete all outliers. In an insider-threat project, unusually high activity may be the signal we want to investigate. Keep unusual observations and analyse them carefully.